# Regularization Study — Ridge vs. Lasso vs. ElasticNet

When a linear model has **more features than it truly needs**, ordinary least squares (OLS) will happily fit the noise: it assigns nonzero weights to irrelevant features just because doing so shaves a little off the training error. That inflates **variance** and hurts generalization.

**Regularization** fights back by adding a penalty on the size of the coefficients to the loss. This trades a little **bias** for a large drop in **variance**. Three classic penalties:

| Model | Penalty added to loss | Effect on coefficients |
|-------|----------------------|------------------------|
| **Ridge** (L2) | $\lambda\lVert w\rVert_2^2$ | shrinks all toward 0, but **none reach exactly 0** |
| **Lasso** (L1) | $\lambda\lVert w\rVert_1$ | drives many coefficients **exactly to 0** (feature selection) |
| **ElasticNet** | $\lambda\big(\rho\lVert w\rVert_1 + (1-\rho)\lVert w\rVert_2^2\big)$ | a blend: sparse *and* stable with correlated features |

The story of this notebook: we build a dataset where **only a few features are informative** and the rest are pure noise (plus some correlated copies). We then watch **Lasso zero out the junk** while Ridge merely shrinks it, and we compare test error across all four models.

In [ ]:
import numpy as np                                  # array math, RNG
import pandas as pd                                 # tidy tables for the coefficient/score summaries
import matplotlib.pyplot as plt                     # all plotting

from sklearn.datasets import make_regression        # synthesize a regression problem with known structure
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler    # standardize features (crucial when penalizing weights)
from sklearn.linear_model import (
    LinearRegression,   # OLS baseline (no penalty)
    Ridge,              # L2 penalty
    Lasso,              # L1 penalty
    ElasticNet,         # L1 + L2 blend
    lasso_path,         # efficient solver for the full coefficient-vs-alpha path
)
from sklearn.metrics import mean_squared_error, r2_score

# One global seed for everything reproducible. Every sklearn call below also takes an
# explicit random_state, so the notebook is deterministic run-to-run.
SEED = 42
np.random.seed(SEED)

plt.rcParams["figure.dpi"] = 100  # crisper inline figures

## 1. Synthesize a dataset with a clear story

We want the regularizers to have something obvious to do. So we build a design matrix with three kinds of columns:

1. **Informative features** — genuinely drive the target (via `n_informative` in `make_regression`).
2. **Noise features** — the remaining `n_features - n_informative` columns are random and have **zero true weight**. A good model should ignore them.
3. **Correlated features** — we manually append a couple of near-duplicates of informative columns. Correlated predictors are exactly where Lasso becomes unstable (it tends to pick one arbitrarily) and where ElasticNet's L2 component helps.

`make_regression` returns `coef=True` so we keep the **ground-truth weights** — invaluable for judging who recovered the real structure.

In [ ]:
# --- Core synthetic regression problem -------------------------------------------------
# n_features = 20 columns total, but only n_informative = 6 actually influence y.
# The other 14 have TRUE coefficient 0 -> they are noise the model must learn to ignore.
# noise= adds Gaussian error to the target so the fit can't be perfect (realistic).
X_base, y, true_coef = make_regression(
    n_samples=300,
    n_features=20,
    n_informative=6,     # only 6 of 20 features are real signal
    noise=12.0,          # observation noise on the target
    coef=True,           # ALSO return the ground-truth weight vector
    random_state=SEED,
)

# --- Inject correlated features -------------------------------------------------------
# Take two genuinely informative columns and append slightly-noised copies of them.
# This creates multicollinearity: several columns carry nearly the same information.
informative_idx = np.flatnonzero(true_coef)              # column indices with nonzero true weight
rng = np.random.default_rng(SEED)
c1 = X_base[:, informative_idx[0]] + 0.01 * rng.standard_normal(X_base.shape[0])  # ~copy of informative col 0
c2 = X_base[:, informative_idx[1]] + 0.01 * rng.standard_normal(X_base.shape[0])  # ~copy of informative col 1
X = np.column_stack([X_base, c1, c2])                    # now 22 columns: 20 base + 2 correlated duplicates

# Ground-truth weights for the 2 appended columns are 0 (y was built WITHOUT them),
# so extend the true-coef vector with two zeros to keep shapes aligned for later comparison.
true_coef_full = np.concatenate([true_coef, [0.0, 0.0]])

feature_names = [f"x{i}" for i in range(X_base.shape[1])] + ["dup_x{}".format(informative_idx[0]),
                                                             "dup_x{}".format(informative_idx[1])]

print(f"Design matrix X: {X.shape[0]} samples x {X.shape[1]} features")
print(f"Truly informative features: {len(informative_idx)}  (indices {list(informative_idx)})")
print(f"Noise + correlated features: {X.shape[1] - len(informative_idx)}")

### Train/test split, then **standardize**

Regularization penalizes coefficient **magnitude**. But a coefficient's magnitude depends on its feature's units — a feature measured in millimetres gets a 1000x larger coefficient than the same feature in metres, and the penalty would clobber it unfairly. So we must put every feature on the **same scale** first.

We fit the `StandardScaler` on the **training set only** (mean 0, std 1 per column) and apply those same statistics to the test set. Fitting the scaler on test data would leak information.

$$x_{\text{scaled}} = \frac{x - \mu_{\text{train}}}{\sigma_{\text{train}}}$$

In [ ]:
# Hold out 30% for testing. random_state makes the split reproducible.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=SEED
)

# Standardize features using TRAIN statistics only, then apply to BOTH splits.
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)   # learn mu/sigma from train AND transform it
X_test = scaler.transform(X_test)          # apply train's mu/sigma to test (no leakage)

# Note: we leave y unscaled. Penalties act on the FEATURE weights, not the target,
# so scaling X is what matters here.
print(f"train: {X_train.shape}, test: {X_test.shape}")
print(f"per-feature train mean ~ 0: {np.allclose(X_train.mean(axis=0), 0, atol=1e-9)}")
print(f"per-feature train std  ~ 1: {np.allclose(X_train.std(axis=0), 1, atol=1e-9)}")

## 2. The four models and their penalties

Every model minimizes squared error, but three of them add a penalty term. Writing $\text{RSS}(w) = \lVert y - Xw\rVert_2^2$:

**OLS** (no penalty) — free to use every feature:
$$\min_w \; \text{RSS}(w)$$

**Ridge (L2)** — penalizes the *squared* magnitude. Shrinks coefficients smoothly toward 0 but never exactly to 0:
$$\min_w \; \text{RSS}(w) + \lambda\lVert w\rVert_2^2, \qquad \lVert w\rVert_2^2 = \sum_j w_j^2$$

**Lasso (L1)** — penalizes the *absolute* magnitude. Its geometry (see the last section) forces many coefficients to become **exactly 0**, performing automatic feature selection:
$$\min_w \; \text{RSS}(w) + \lambda\lVert w\rVert_1, \qquad \lVert w\rVert_1 = \sum_j |w_j|$$

**ElasticNet** — a convex blend of the two, governed by the mixing ratio $\rho$ (`l1_ratio`):
$$\min_w \; \text{RSS}(w) + \lambda\Big(\rho\lVert w\rVert_1 + (1-\rho)\lVert w\rVert_2^2\Big)$$

> In sklearn the penalty strength is called `alpha` (our $\lambda$). Larger `alpha` = stronger penalty = smaller coefficients.

In [ ]:
# Fit all four models on the SAME standardized training data.
# max_iter is set generously so the coordinate-descent solvers (Lasso/ElasticNet)
# fully converge -> no ConvergenceWarning.
models = {
    "OLS":        LinearRegression(),
    "Ridge":      Ridge(alpha=10.0, random_state=SEED),
    "Lasso":      Lasso(alpha=1.0, max_iter=100000, random_state=SEED),
    "ElasticNet": ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=100000, random_state=SEED),
}

fitted = {}
for name, model in models.items():
    model.fit(X_train, y_train)   # each model learns its own coefficient vector
    fitted[name] = model

# Collect every model's learned coefficients into one tidy table alongside the ground truth.
coef_table = pd.DataFrame({"true": true_coef_full}, index=feature_names)
for name, model in fitted.items():
    coef_table[name] = model.coef_

# Round for readability; tiny values print as 0.0 which is exactly the point for Lasso.
print(coef_table.round(2).to_string())

## 3. Key demonstration — Lasso creates sparsity, Ridge does not

Now we **count nonzero coefficients** for each model. Because floating-point solvers may leave a coefficient at something like `1e-15` rather than a clean `0.0`, we treat anything with absolute value below a small tolerance as zero.

Recall the dataset has only **6 truly informative features** out of 22. A good feature selector should keep roughly that many and discard the rest.

In [ ]:
TOL = 1e-6  # coefficients with |w| < TOL are considered effectively zero

def count_nonzero(coef, tol=TOL):
    """How many coefficients are meaningfully different from zero."""
    return int(np.sum(np.abs(coef) > tol))

n_features_total = X_train.shape[1]
print(f"Total features: {n_features_total}   (only 6 are truly informative)\n")
for name, model in fitted.items():
    nz = count_nonzero(model.coef_)
    zeros = n_features_total - nz
    print(f"{name:<11}: {nz:2d} nonzero,  {zeros:2d} driven to zero")

# Pull out the two headline numbers the story hinges on.
lasso_nonzero = count_nonzero(fitted["Lasso"].coef_)
ridge_nonzero = count_nonzero(fitted["Ridge"].coef_)
print(f"\n>>> Lasso keeps {lasso_nonzero} features; Ridge keeps all {ridge_nonzero}.")
print(">>> L1 performs feature selection; L2 only shrinks.")

### Coefficient bar chart across the four models

The clearest way to *see* sparsity: plot every model's coefficient for every feature. Watch how **Lasso's bars collapse to the baseline** for the noise/correlated features, while Ridge's bars stay small-but-nonzero everywhere.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))

x = np.arange(n_features_total)   # one group of bars per feature
width = 0.2                        # width of each individual bar within a group

# Draw the four models side-by-side within each feature's slot.
for i, (name, model) in enumerate(fitted.items()):
    # offset each model's bars so they sit next to each other, centered on the tick
    offset = (i - 1.5) * width
    ax.bar(x + offset, model.coef_, width, label=name)

ax.axhline(0, color="k", linewidth=0.8)                 # baseline: coefficient = 0
ax.set_xticks(x)
ax.set_xticklabels(feature_names, rotation=90, fontsize=8)
ax.set_ylabel("coefficient value")
ax.set_title("Learned coefficients per feature — Lasso zeros out the irrelevant ones")
ax.legend()

# Shade the columns that are TRULY informative so the eye can check who recovered them.
for idx in informative_idx:
    ax.axvspan(idx - 0.5, idx + 0.5, color="gold", alpha=0.15)

plt.tight_layout()
plt.show()
print("Gold bands mark the 6 genuinely informative features (nonzero true weight).")

### The Lasso coefficient **path**

The single most instructive plot in regularization: sweep the penalty strength `alpha` from small to large and trace each coefficient. As `alpha` grows, the penalty tightens and coefficients are pulled to zero **one after another** — and once a coefficient hits zero under L1, it *stays* there. At large enough `alpha`, every coefficient is zero (the model predicts only the mean).

`lasso_path` computes this efficiently over a whole grid of `alpha` values in one call.

In [ ]:
# Build an explicit log-spaced grid of penalty strengths, from strong to weak.
# alpha_max is (up to scaling) the smallest penalty that zeros ALL coefficients;
# below it they switch on one by one.
n_samples = X_train.shape[0]
alpha_max = np.max(np.abs(X_train.T @ (y_train - y_train.mean()))) / n_samples
alpha_grid = np.logspace(np.log10(alpha_max), np.log10(alpha_max * 1e-3), 100)

# Compute the full path: coefs has shape (n_features, n_alphas). Passing an EXPLICIT
# `alphas` grid keeps the solver deterministic and sidesteps the n_alphas deprecation.
alphas, coefs, _ = lasso_path(X_train, y_train, alphas=alpha_grid)

fig, ax = plt.subplots(figsize=(10, 6))

# Plot each feature's coefficient as a function of alpha. We use a LOG x-axis because
# alpha spans several orders of magnitude. Informative features are drawn bold+colored;
# noise features are thin grey so the contrast is obvious.
for j in range(coefs.shape[0]):
    if j in informative_idx:
        ax.plot(alphas, coefs[j], linewidth=2.2, label=feature_names[j])
    else:
        ax.plot(alphas, coefs[j], linewidth=0.8, color="grey", alpha=0.5)

ax.set_xscale("log")
ax.set_xlim(ax.get_xlim()[::-1])   # reverse x-axis: strong penalty (large alpha) on the LEFT
ax.set_xlabel("alpha  (penalty strength, log scale, decreasing to the right)")
ax.set_ylabel("coefficient value")
ax.set_title("Lasso path — coefficients shrink to EXACTLY zero as alpha grows")
ax.axhline(0, color="k", linewidth=0.8)
ax.legend(title="informative features", fontsize=8, ncol=2)
plt.tight_layout()
plt.show()
print("Left = heavy penalty (all coefs ~0). Moving right, informative features (colored)")
print("switch on first; grey noise features stay near zero the longest.")

## 4. Test-error comparison

Sparsity is only useful if it doesn't wreck predictive accuracy. We evaluate each model on the held-out test set with two standard regression metrics:

- **RMSE** — root mean squared error, in the target's own units (lower is better).
- **$R^2$** — fraction of target variance explained (1.0 is perfect).

$$\text{RMSE} = \sqrt{\tfrac{1}{n}\textstyle\sum_i (y_i - \hat y_i)^2}, \qquad R^2 = 1 - \frac{\sum_i (y_i-\hat y_i)^2}{\sum_i (y_i - \bar y)^2}$$

In [ ]:
rows = []
for name, model in fitted.items():
    # Evaluate on BOTH splits to expose overfitting (train much better than test).
    pred_tr = model.predict(X_train)
    pred_te = model.predict(X_test)
    rows.append({
        "model": name,
        "train_RMSE": np.sqrt(mean_squared_error(y_train, pred_tr)),
        "test_RMSE":  np.sqrt(mean_squared_error(y_test,  pred_te)),
        "train_R2":   r2_score(y_train, pred_tr),
        "test_R2":    r2_score(y_test,  pred_te),
        "nonzero":    count_nonzero(model.coef_),
    })

scores = pd.DataFrame(rows).set_index("model")
print(scores.round(3).to_string())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))

names = list(fitted.keys())
colors = ["#888888", "#4c72b0", "#c44e52", "#55a868"]  # one color per model

# Left: test RMSE (lower is better).
bars0 = ax[0].bar(names, scores["test_RMSE"], color=colors)
ax[0].set_title("Test RMSE (lower is better)")
ax[0].set_ylabel("RMSE")
for b, v in zip(bars0, scores["test_RMSE"]):
    ax[0].text(b.get_x() + b.get_width() / 2, v, f"{v:.2f}", ha="center", va="bottom", fontsize=9)

# Right: test R2 (higher is better).
bars1 = ax[1].bar(names, scores["test_R2"], color=colors)
ax[1].set_title("Test $R^2$ (higher is better)")
ax[1].set_ylabel("$R^2$")
ax[1].set_ylim(0, 1)
for b, v in zip(bars1, scores["test_R2"]):
    ax[1].text(b.get_x() + b.get_width() / 2, v, f"{v:.3f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()

## 5. Why does L1 induce sparsity but L2 does not?

It comes down to the **geometry of the penalty**. Constrained optimization gives the cleanest intuition. Minimizing RSS subject to a budget on the coefficients is equivalent to the penalized forms above:

$$\text{Lasso: } \min_w \text{RSS}(w) \;\text{ s.t. } \lVert w\rVert_1 \le t \qquad\qquad \text{Ridge: } \min_w \text{RSS}(w)\;\text{ s.t. } \lVert w\rVert_2^2 \le t$$

The RSS objective has **elliptical contours** centered on the OLS solution. The regularizer restricts $w$ to a **constraint region**, and the solution is the first point where the growing ellipse touches that region:

- The **L1** region $\lVert w\rVert_1 \le t$ is a **diamond** (rotated square) — it has sharp **corners on the axes**. An expanding ellipse is very likely to first touch a corner, and a corner means one coordinate is **exactly zero**. That is sparsity.
- The **L2** region $\lVert w\rVert_2^2 \le t$ is a **circle** — smooth, no corners. The tangent point almost never lands exactly on an axis, so coefficients get **small but stay nonzero**.

The cell below draws these two constraint regions with a shared set of RSS contours so you can see the diamond's corner grabbing the axis.

In [ ]:
# A 2-D cartoon (two coefficients w1, w2) illustrating the corner argument.
fig, axes = plt.subplots(1, 2, figsize=(11, 5.2))

# Pretend the unconstrained (OLS) optimum sits off-axis; RSS contours are ellipses around it.
w_ols = np.array([2.2, 1.4])
w1 = np.linspace(-1.0, 3.5, 400)
w2 = np.linspace(-1.0, 3.0, 400)
W1, W2 = np.meshgrid(w1, w2)
# Elliptical RSS surface: a positive-definite quadratic centered at w_ols (with a tilt term).
RSS = 1.0 * (W1 - w_ols[0]) ** 2 + 2.0 * (W2 - w_ols[1]) ** 2 + 0.8 * (W1 - w_ols[0]) * (W2 - w_ols[1])

t = 1.5  # the coefficient budget (radius of the constraint region)

for ax, kind in zip(axes, ["L1 (Lasso): diamond", "L2 (Ridge): circle"]):
    ax.contour(W1, W2, RSS, levels=12, cmap="Blues", linewidths=0.8)   # RSS ellipses
    ax.plot(*w_ols, "x", color="navy", markersize=10, markeredgewidth=2, label="OLS optimum")
    ax.axhline(0, color="grey", lw=0.6); ax.axvline(0, color="grey", lw=0.6)

    if kind.startswith("L1"):
        # Diamond |w1| + |w2| = t : its four vertices lie ON the axes (corners).
        diamond = np.array([[t, 0], [0, t], [-t, 0], [0, -t], [t, 0]])
        ax.plot(diamond[:, 0], diamond[:, 1], color="crimson", lw=2, label=r"$\|w\|_1 \leq t$")
        # The ellipse first meets the diamond at the top vertex (0, t): w1 = 0 -> sparse!
        ax.plot(0, t, "o", color="crimson", markersize=9)
        ax.annotate("touches at a CORNER\n(w1 = 0 exactly)", (0, t),
                    textcoords="offset points", xytext=(15, 10), fontsize=9, color="crimson")
    else:
        # Circle w1^2 + w2^2 = t^2 : smooth boundary, no corners.
        theta = np.linspace(0, 2 * np.pi, 300)
        ax.plot(t * np.cos(theta), t * np.sin(theta), color="crimson", lw=2, label=r"$\|w\|_2 \leq t$")
        # Tangent point is off-axis -> both coefficients nonzero (just shrunk).
        w_touch = w_ols / np.linalg.norm(w_ols) * t
        ax.plot(*w_touch, "o", color="crimson", markersize=9)
        ax.annotate("touches OFF-axis\n(both nonzero)", w_touch,
                    textcoords="offset points", xytext=(10, 10), fontsize=9, color="crimson")

    ax.set_title(kind); ax.set_xlabel("w1"); ax.set_ylabel("w2")
    ax.set_aspect("equal"); ax.legend(loc="lower left", fontsize=8)

plt.tight_layout()
plt.show()

## 6. Takeaways — the bias-variance / feature-selection trade-off

- **OLS** uses every feature, including the 14 noise columns and 2 correlated duplicates. It fits the training set best but its coefficients are noisy (look at how it split weight between `x5` and its duplicate `dup_x5` with huge opposite signs) — the classic **high-variance** failure mode.
- **Ridge (L2)** shrinks all coefficients toward zero, taming the variance, but it **keeps every feature** — no sparsity. Great when you believe *many* features each contribute a little, and especially stable under correlation (it spreads weight across correlated copies rather than picking one).
- **Lasso (L1)** drives the junk coefficients **exactly to zero**, yielding a sparse, interpretable model that essentially recovers the small set of informative features. The cost is a bit more bias, and instability when features are highly correlated (it arbitrarily keeps one of a correlated group).
- **ElasticNet** blends both: it inherits Lasso's sparsity while L2 keeps it stable across correlated features, often the safest default when you have many correlated predictors. (Here its stronger effective shrinkage costs some accuracy — a reminder that `alpha`/`l1_ratio` should be tuned by cross-validation.)

**The core trade-off:** regularization deliberately adds a little **bias** (by shrinking/zeroing coefficients) to buy a reduction in **variance**. On a problem like this one — few real signals hidden among many noise features — L1's sparsity additionally hands you free feature selection with essentially no loss in test accuracy.